## Автоматическое создание датасета кропов деревьев

Этот ноутбук автоматизирует процесс создания датасета из обрезанных изображений деревьев и кустарников для дальнейшего обучения моделей классификации:

### Основные компоненты:
- **Grounding DINO** - для обнаружения деревьев и кустарников на изображениях
- **PIL/OpenCV** - для обработки и обрезки изображений
- **CSV манифест** - для отслеживания метаданных каждого кропа

### Процесс работы:
1. **Обнаружение объектов**: Grounding DINO находит деревья и кустарники на исходных изображениях
2. **Обрезка изображений**: Извлекает bounding box'ы и создает отдельные кропы
3. **Сохранение метаданных**: Записывает информацию о каждом кропе (координаты, класс, скор)
4. **Создание манифеста**: Генерирует CSV файл с полной информацией о датасете

### Возможности:
- Обнаружение деревьев (`"a tree"`) и кустарников (`"a shrub"`)
- Настраиваемые пороги детекции (threshold=0.40)
- Ограничение количества объектов на класс (MAX_PER_CLASS=5)
- Автоматическое именование файлов с метаданными
- Создание структурированного датасета для обучения

### Выходные данные:
- Папка с кропами: `data/crops_dataset/crops/`
- Манифест: `data/crops_dataset/manifest.csv` с колонками: orig_path, crop_path, obj_index, class, score, координаты

## Применение:
Идеально подходит для подготовки данных для обучения классификаторов видов деревьев, анализа состояния растительности.

In [ ]:
import os
import re
import csv
import json
import random
from pathlib import Path
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from huggingface_hub import login
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
)

from tqdm.notebook import tqdm

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

In [ ]:
GROUNDING_DINO_ID = "IDEA-Research/grounding-dino-tiny"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def load_all_models() -> Dict[str, Any]:
    models = {}
    print("1/3: Grounding DINO...")
    models['dino_processor'] = AutoProcessor.from_pretrained(GROUNDING_DINO_ID)
    models['dino_model'] = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_ID).to(DEVICE)
    models['dino_model'].eval()
    
    return models

In [ ]:
models = load_all_models()

In [ ]:
def detect_objects(
    dino_model,
    dino_processor,
    image: Image.Image
) -> Dict[str, Any]:
    text_labels = [["a tree", "a shrub"]]
    inputs = dino_processor(
        images=image, text=text_labels, return_tensors="pt"
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = dino_model(**inputs)
        
    results = dino_processor.post_process_grounded_object_detection(
        outputs,
        threshold=0.40,
        text_threshold=0.25,
        target_sizes=[image.size[::-1]]
    )
    res = results[0]

    if torch.is_tensor(res.get("boxes", None)):
        boxes = res.get("boxes", torch.empty((0, 4))).detach().cpu()
    else:
        boxes = torch.tensor(res.get("boxes", []), dtype=torch.float32)

    if torch.is_tensor(res.get("scores", None)):
        scores = res.get("scores", torch.empty((0,))).detach().cpu()
    else:
        scores = torch.tensor(res.get("scores", []), dtype=torch.float32)

    text_labels = res.get("text_labels", [])

    if boxes.numel() == 0:
        boxes = torch.empty((0,4), dtype=torch.float32)
    if scores.numel() == 0:
        scores = torch.empty((0,), dtype=torch.float32)

    return {
        "boxes": boxes.cpu(),
        "scores": scores.cpu(),
        "text_labels": list(text_labels)
    }

In [ ]:
def detect_multiple_objects(
    dino_model,
    dino_processor,
    image: Image.Image,
    target_classes: List[str] = ["tree", "shrub"],
    max_objects_per_class: int = 5
) -> Dict[str, Any]:
    all_boxes = []
    all_scores = []
    all_labels = []

    for class_name in target_classes:
        text_labels = [[f"a {class_name}"]]
        
        inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = dino_model(**inputs)
            
        results = dino_processor.post_process_grounded_object_detection(
            outputs,
            threshold=0.40,
            text_threshold=0.25,
            target_sizes=[image.size[::-1]]
        )[0]

        num_found = len(results['boxes'])
        if num_found > 0:
            num_to_keep = min(num_found, max_objects_per_class)
            
            all_boxes.append(results['boxes'][:num_to_keep])
            all_scores.append(results['scores'][:num_to_keep])
            all_labels.extend([class_name] * num_to_keep)
        
    if not all_boxes:
        return {
            "boxes": torch.empty((0, 4), device=DEVICE),
            "scores": torch.empty(0, device=DEVICE),
            "text_labels": []
        }

    final_boxes = torch.cat(all_boxes, dim=0)
    final_scores = torch.cat(all_scores, dim=0)
    
    merged_results = {
        "scores": final_scores,
        "boxes": final_boxes,
        "text_labels": all_labels
    }
    
    return merged_results

In [ ]:
INPUT_DIR = Path("data/dataset")
OUTPUT_DIR = Path("data/crops_dataset")
CROPS_SUBDIR = OUTPUT_DIR / "crops"
MAX_PER_CLASS = 5
TARGET_CLASSES = ["tree", "shrub"]
SCORE_FMT = "{:.3f}"

In [ ]:
CROPS_SUBDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def _sanitize_filename(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-_\. ]", "_", s)
    return s

In [ ]:
def _to_numpy_boxes_scores(res: dict):
    boxes = res.get("boxes", None)
    scores = res.get("scores", None)
    labels = res.get("text_labels", []) or []

    if boxes is None:
        boxes_np = np.zeros((0,4), dtype=np.float32)
    elif torch.is_tensor(boxes):
        boxes_np = boxes.detach().cpu().numpy()
    else:
        boxes_np = np.array(boxes, dtype=np.float32)

    if scores is None:
        scores_np = np.zeros((boxes_np.shape[0],), dtype=np.float32)
    elif torch.is_tensor(scores):
        scores_np = scores.detach().cpu().numpy()
    else:
        scores_np = np.array(scores, dtype=np.float32)

    if len(labels) < boxes_np.shape[0]:
        labels = labels + [""] * (boxes_np.shape[0] - len(labels))

    return boxes_np, scores_np, list(labels)

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

In [ ]:
files = [p for p in sorted(INPUT_DIR.iterdir()) if p.suffix.lower() in IMG_EXTS]

manifest_rows = []
row_header = ["orig_path","crop_path","obj_index","class","score","x1","y1","x2","y2","orig_width","orig_height"]

for img_path in tqdm(files, desc="Processing images"):
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Failed to open {img_path}: {e}")
        continue

    W, H = img.size

    try:
        # res = detect_multiple_objects(
        #     models['dino_model'],
        #     models['dino_processor'],
        #     img, 
        #     target_classes=TARGET_CLASSES, 
        #     max_objects_per_class=MAX_PER_CLASS
        # )
        res = detect_objects(models['dino_model'], models['dino_processor'], img)
    except Exception as e:
        print(f"Detection failed for {img_path}: {e}")
        continue

    boxes_np, scores_np, labels = _to_numpy_boxes_scores(res)

    if boxes_np.shape[0] == 0:
        continue

    for i in range(boxes_np.shape[0]):
        x1, y1, x2, y2 = boxes_np[i].astype(int).tolist()
        x1 = max(0, min(x1, W-1))
        x2 = max(0, min(x2, W))
        y1 = max(0, min(y1, H-1))
        y2 = max(0, min(y2, H))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))

        label = labels[i] if i < len(labels) else ""
        score = float(scores_np[i]) if i < len(scores_np) else 0.0

        base = img_path.stem
        label_safe = _sanitize_filename(label or "obj")
        crop_name = f"{base}_obj{i+1}_{label_safe}_{SCORE_FMT.format(score)}.jpg"
        crop_path = CROPS_SUBDIR / crop_name

        try:
            crop.save(crop_path, quality=95)
        except Exception as e:
            crop_path = CROPS_SUBDIR / (crop_name.replace(".jpg", ".png"))
            crop.save(crop_path)

        manifest_rows.append({
            "orig_path": str(img_path),
            "crop_path": str(crop_path),
            "obj_index": i+1,
            "class": label,
            "score": score,
            "x1": int(x1),
            "y1": int(y1),
            "x2": int(x2),
            "y2": int(y2),
            "orig_width": int(W),
            "orig_height": int(H)
        })

In [ ]:
manifest_file = OUTPUT_DIR / "manifest.csv"

with open(manifest_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row_header)
    writer.writeheader()
    for r in manifest_rows:
        writer.writerow(r)

print(f"Done. Saved {len(manifest_rows)} crops to {CROPS_SUBDIR} and manifest to {manifest_file}")